# RNN: Mortality Prediction on MIMIC-III (Baseline — No code_mapping)

This notebook runs the RNN model for mortality prediction **without** `code_mapping`.
Raw ICD-9 and NDC codes are used as-is for the embedding vocabulary.

**Model:** RNN (GRU)  
**Task:** In-hospital mortality prediction  
**Dataset:** Synthetic MIMIC-III (`dev=False`)

## Step 1: Load the MIMIC-III Dataset

In [4]:
# !pip install git+https://github.com/lookman-olowo/PyHealth.git@refactor/grasp-model
!pip install ipywidgets

# ! pip uninstall pyhealth

  Using cached ipywidgets-8.1.8-py3-none-any.whl.metadata (2.4 kB)
  Using cached widgetsnbextension-4.0.15-py3-none-any.whl.metadata (1.6 kB)
  Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl.metadata (20 kB)
Using cached ipywidgets-8.1.8-py3-none-any.whl (139 kB)
Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl (914 kB)
Using cached widgetsnbextension-4.0.15-py3-none-any.whl (2.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ipywidgets]


In [5]:
import tempfile

from pyhealth.datasets import MIMIC3Dataset

base_dataset = MIMIC3Dataset(
    root="/home/lolowo2",
    tables=["DIAGNOSES_ICD", "PROCEDURES_ICD", "PRESCRIPTIONS"],
    cache_dir=tempfile.TemporaryDirectory().name,
    dev=False,
)

base_dataset.stats()

No config path provided, using default config
Initializing mimic3 dataset from /home/lolowo2 (dev mode: False)
Using provided cache_dir: /tmp/tmphrbg7b9w/4f338cfd-b388-50e8-9d9c-fa4872e51b6c
No cached event dataframe found. Creating: /tmp/tmphrbg7b9w/4f338cfd-b388-50e8-9d9c-fa4872e51b6c/global_event_df.parquet
Scanning table: patients from /home/lolowo2/PATIENTS.csv.gz
Scanning table: admissions from /home/lolowo2/ADMISSIONS.csv.gz
Scanning table: icustays from /home/lolowo2/ICUSTAYS.csv.gz
Scanning table: diagnoses_icd from /home/lolowo2/DIAGNOSES_ICD.csv.gz
Joining with table: /home/lolowo2/ADMISSIONS.csv.gz
Scanning table: procedures_icd from /home/lolowo2/PROCEDURES_ICD.csv.gz
Joining with table: /home/lolowo2/ADMISSIONS.csv.gz
Scanning table: prescriptions from /home/lolowo2/PRESCRIPTIONS.csv.gz
Joining with table: /home/lolowo2/ADMISSIONS.csv.gz
Caching event dataframe to /tmp/tmphrbg7b9w/4f338cfd-b388-50e8-9d9c-fa4872e51b6c/global_event_df.parquet...
Dataset: mimic3
Dev mode: Fa

## Step 2: Define the Mortality Prediction Task

In [6]:
from pyhealth.tasks import MortalityPredictionMIMIC3

task = MortalityPredictionMIMIC3()

samples = base_dataset.set_task(task)

print(f"Generated {len(samples)} samples")
print(f"\nInput schema: {samples.input_schema}")
print(f"Output schema: {samples.output_schema}")

Setting task MortalityPredictionMIMIC3 for mimic3 base dataset...
Task cache paths: task_df=/tmp/tmphrbg7b9w/4f338cfd-b388-50e8-9d9c-fa4872e51b6c/tasks/MortalityPredictionMIMIC3_187a839c-4f67-585a-bbb3-355429e27594/task_df.ld, samples=/tmp/tmphrbg7b9w/4f338cfd-b388-50e8-9d9c-fa4872e51b6c/tasks/MortalityPredictionMIMIC3_187a839c-4f67-585a-bbb3-355429e27594/samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld
Applying task transformations on data with 1 workers...
Detected Jupyter notebook environment, setting num_workers to 1
Single worker mode, processing sequentially
Worker 0 started processing 46520 patients. (Polars threads: 16)


  0%|          | 0/46520 [00:00<?, ?it/s]

Rank 0 inferred the following `['bytes']` data format.


100%|██████████| 46520/46520 [01:24<00:00, 550.54it/s]

Worker 0 finished processing patients.
Fitting processors on the dataset...


...

  0%|          | 0/9583 [00:00<?, ?it/s]

Rank 0 inferred the following `['str', 'str', 'no_header_tensor:18', 'no_header_tensor:18', 'no_header_tensor:18', 'no_header_tensor:1']` data format.


100%|██████████| 9583/9583 [00:00<00:00, 18893.74it/s]

Worker 0 finished processing samples.


Cached processed samples to /tmp/tmphrbg7b9w/4f338cfd-b388-50e8-9d9c-fa4872e51b6c/tasks/MortalityPredictionMIMIC3_187a839c-4f67-585a-bbb3-355429e27594/samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld
Generated 9583 samples

Input schema: {'conditions': 'sequence', 'procedures': 'sequence', 'drugs': 'sequence'}
Output schema: {'mortality': 'binary'}


## Step 3: Dataset Statistics

In [7]:
print("Sample structure:")
print(samples[0])

print("\n" + "=" * 50)
print("Processor Vocabulary Sizes:")
print("=" * 50)
for key, proc in samples.input_processors.items():
    if hasattr(proc, 'code_vocab'):
        print(f"{key}: {len(proc.code_vocab)} codes (including <pad>, <unk>)")

mortality_count = sum(float(s.get("mortality", 0)) for s in samples)
print(f"\nTotal samples: {len(samples)}")
print(f"Mortality rate: {mortality_count / len(samples) * 100:.2f}%")
print(f"Positive samples: {int(mortality_count)}")
print(f"Negative samples: {len(samples) - int(mortality_count)}")

Sample structure:
{'hadm_id': '164713', 'patient_id': '10004', 'conditions': tensor([ 2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16]), 'procedures': tensor([2, 3, 4, 5, 6, 7, 8]), 'drugs': tensor([ 2,  3,  4,  5,  6,  7,  3,  8,  9, 10, 11, 12,  6, 13, 14, 15, 15,  3,
         3,  3, 16, 17, 18, 19,  8, 20, 21,  3, 21, 22,  3,  4, 23, 24,  3, 25,
        25, 26, 27,  3, 15, 26, 26, 26, 28, 29, 28, 30,  5, 31, 32, 30, 33, 33,
        34, 35, 11, 33, 36, 35,  8,  8, 35,  4,  4,  9, 10, 35, 35, 37, 35, 14,
        17, 38, 39,  4,  4, 12,  3, 10, 35, 28, 28,  6, 29, 28, 28, 40, 41, 38,
        19,  4, 38, 38,  6,  4, 42, 10, 43,  4, 43, 44,  3, 45, 43, 43, 43,  4,
         9, 35, 46, 45, 43,  6, 47,  6, 48, 49, 50, 51, 52,  6,  4,  9, 38, 53,
         3,  6,  6,  6,  4, 50, 41, 54,  6,  6,  6,  6,  6,  6, 47, 55, 56, 51,
        57,  6,  6,  6,  6,  6,  6, 35, 35, 53, 57,  6,  6,  6, 35, 19, 58, 35,
        59, 59, 59,  6,  6,  6, 47, 54, 53, 43, 42, 29, 28, 28, 60, 31, 61, 11,

## Step 4: Split the Dataset

In [8]:
from pyhealth.datasets import split_by_patient

train_dataset, val_dataset, test_dataset = split_by_patient(
    samples, [0.8, 0.1, 0.1], seed=42
)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")

Training samples: 7711
Validation samples: 909
Test samples: 963


## Step 5: Create Data Loaders

In [9]:
from pyhealth.datasets import get_dataloader

train_dataloader = get_dataloader(train_dataset, batch_size=32, shuffle=True)
val_dataloader = get_dataloader(val_dataset, batch_size=32, shuffle=False)
test_dataloader = get_dataloader(test_dataset, batch_size=32, shuffle=False)

print(f"Training batches: {len(train_dataloader)}")
print(f"Validation batches: {len(val_dataloader)}")
print(f"Test batches: {len(test_dataloader)}")

Training batches: 241
Validation batches: 29
Test batches: 31


## Step 6: Initialize the RNN Model

In [10]:
from pyhealth.models import RNN

model = RNN(
    dataset=samples,
    embedding_dim=128,
    hidden_dim=128,
)

print(f"Model initialized with {sum(p.numel() for p in model.parameters()):,} parameters")
print(f"\nModel architecture:")
print(model)

Model initialized with 1,325,441 parameters

Model architecture:
RNN(
  (embedding_model): EmbeddingModel(embedding_layers=ModuleDict(
    (conditions): Embedding(4102, 128, padding_idx=0)
    (procedures): Embedding(1304, 128, padding_idx=0)
    (drugs): Embedding(2624, 128, padding_idx=0)
  ))
  (rnn): ModuleDict(
    (conditions): RNNLayer(
      (dropout_layer): Dropout(p=0.5, inplace=False)
      (rnn): GRU(128, 128, batch_first=True)
    )
    (procedures): RNNLayer(
      (dropout_layer): Dropout(p=0.5, inplace=False)
      (rnn): GRU(128, 128, batch_first=True)
    )
    (drugs): RNNLayer(
      (dropout_layer): Dropout(p=0.5, inplace=False)
      (rnn): GRU(128, 128, batch_first=True)
    )
  )
  (fc): Linear(in_features=384, out_features=1, bias=True)
)


## Step 7: Train the Model

In [11]:
from pyhealth.trainer import Trainer

trainer = Trainer(
    model=model,
    metrics=["roc_auc", "pr_auc", "accuracy", "f1"],
)

trainer.train(
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    epochs=50,
    monitor="roc_auc",
    optimizer_params={"lr": 1e-3},
)

RNN(
  (embedding_model): EmbeddingModel(embedding_layers=ModuleDict(
    (conditions): Embedding(4102, 128, padding_idx=0)
    (procedures): Embedding(1304, 128, padding_idx=0)
    (drugs): Embedding(2624, 128, padding_idx=0)
  ))
  (rnn): ModuleDict(
    (conditions): RNNLayer(
      (dropout_layer): Dropout(p=0.5, inplace=False)
      (rnn): GRU(128, 128, batch_first=True)
    )
    (procedures): RNNLayer(
      (dropout_layer): Dropout(p=0.5, inplace=False)
      (rnn): GRU(128, 128, batch_first=True)
    )
    (drugs): RNNLayer(
      (dropout_layer): Dropout(p=0.5, inplace=False)
      (rnn): GRU(128, 128, batch_first=True)
    )
  )
  (fc): Linear(in_features=384, out_features=1, bias=True)
)
Metrics: ['roc_auc', 'pr_auc', 'accuracy', 'f1']
Device: cuda

Training:
Batch size: 32
Optimizer: <class 'torch.optim.adam.Adam'>
Optimizer params: {'lr': 0.001}
Weight decay: 0.0
Max grad norm: None
Val dataloader: <torch.utils.data.dataloader.DataLoader object at 0x7c2c8c2b7d90>
Monitor:

Epoch 0 / 50:   0%|          | 0/241 [00:00<?, ?it/s]

--- Train epoch-0, step-241 ---
loss: 0.3894


Evaluation: 100%|██████████| 29/29 [00:02<00:00, 11.51it/s]

--- Eval epoch-0, step-241 ---
roc_auc: 0.6141
pr_auc: 0.1667
accuracy: 0.8856
f1: 0.0000
loss: 0.3529
New best roc_auc score (0.6141) at epoch-0, step-241



...

--- Eval epoch-48, step-11809 ---
roc_auc: 0.5412
pr_auc: 0.1489
accuracy: 0.8482
f1: 0.1585
loss: 1.3498



Epoch 49 / 50:   0%|          | 0/241 [00:00<?, ?it/s]

--- Train epoch-49, step-12050 ---
loss: 0.0273


Evaluation: 100%|██████████| 29/29 [00:02<00:00, 11.61it/s]

--- Eval epoch-49, step-12050 ---
roc_auc: 0.5416
pr_auc: 0.1452
accuracy: 0.8493
f1: 0.1274
loss: 1.3399
Loaded best model


## Step 8: Evaluate on Test Set

In [12]:
test_results = trainer.evaluate(test_dataloader)

print("\n" + "=" * 50)
print("Test Set Performance (NO code_mapping)")
print("=" * 50)
for metric, value in test_results.items():
    print(f"{metric}: {value:.4f}")

Evaluation: 100%|██████████| 31/31 [00:02<00:00, 12.18it/s]


Test Set Performance (NO code_mapping)
roc_auc: 0.6292
pr_auc: 0.1710
accuracy: 0.8816
f1: 0.0000
loss: 0.3475
